# ELIZA kontra LLM — gdzie jest „rozumienie"?

**Podstawy AI · Laboratorium 2**

Na poprzednich zajęciach zbudowałeś ELIZĘ i znalazłeś miejsca, w których
jej złudzenie pęka. Dziś te same zdania trafią do dużego modelu językowego
(LLM) i sprawdzimy: **które pęknięcia LLM naprawia, a które zostają?**

**Plan zajęć**

| Sekcja | Co robisz |
|---|---|
| 1. Jak rozmawia się z LLM przez API | czytasz (5 min) |
| 2. Przygotowanie | klucz, klient, test połączenia |
| 3. Twoja ELIZA wraca | wklejasz kod z Lab 1 |
| 4. Ta sama rozmowa, dwa silniki | Zadanie 8 |
| 5. Łamiące zdania — druga runda | Zadanie 9 |
| 6. Pamięć to lista wiadomości | Zadanie 10 |
| 7. Rola to tekst | Zadanie 11 |
| 8. Refleksja | Zadanie 12 |

**Potrzebujesz:** klucza Gemini API (instrukcję dostałeś przed zajęciami)
i notebooka z Lab 1.

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 1. Jak rozmawia się z LLM przez API

**Czym jest LLM (w pięciu zdaniach).** Duży model językowy to sieć neuronowa
wytrenowana na ogromnych zbiorach tekstu do jednego zadania: przewidywania,
jaki fragment tekstu (*token*) powinien pojawić się dalej. Odpowiedź powstaje
token po tokenie — model za każdym razem wybiera kolejny fragment, a wybór
jest częściowo losowy (*próbkowanie*). Nie ma tu reguł pisanych ręcznie jak
w ELIZIE: wszystko, co model „umie", jest zapisane w miliardach wyuczonych
parametrów. To, czy i w jakim sensie model *rozumie* tekst, jest przedmiotem
sporu — i właśnie dlatego porównujemy go dziś z ELIZĄ.

**Zapytanie do API.** Rozmawiamy z modelem, wysyłając mu **listę wiadomości**.
Każda wiadomość ma rolę:

| Rola | Kto mówi | Przykład |
|---|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać | „Jesteś terapeutą. Odpowiadaj krótko." |
| `user` | użytkownik | „Czuję się samotny." |
| `assistant` | wcześniejsze odpowiedzi modelu | „Od kiedy tak się czujesz?" |

```python
messages = [
    {"role": "system", "content": "Jesteś terapeutą..."},
    {"role": "user",   "content": "Czuję się samotny."},
]
```

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 10 okaże się ważniejszy, niż wygląda.

**Biblioteka.** Użyjemy biblioteki `openai`. Mimo nazwy nie łączymy się
z OpenAI: Gemini udostępnia *zgodny* interfejs, więc wystarczy podać inny
adres serwera (`base_url`) i swój klucz.

## 2. Przygotowanie

Klucz podajesz przez `getpass` — nie pojawi się na ekranie ani w zapisanym
notebooku. **Nigdy nie wklejaj klucza wprost do kodu**: notebook oddajesz
prowadzącemu, a klucz to dostęp do Twojego konta.

Jeśli pracujesz w Google Colab, możesz zamiast tego zapisać klucz
w *Secrets* (ikona klucza po lewej) pod nazwą `GEMINI_API_KEY`.

In [71]:
# !pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import os
import time
from getpass import getpass
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                       # lokalnie: zmienna środowiskowa albo getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Wklej klucz Gemini API: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.8-flash"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "low"             # ogranicza „myślenie" modelu -> szybciej i taniej;
                                  # jeśli dostajesz błąd 400 dotyczący reasoning_effort, ustaw None
print("Klient gotowy. Model:", MODEL)

Klient gotowy. Model: gemini-3.8-flash


In [2]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


Poniższa funkcja `czat` wysyła listę wiadomości i zwraca tekst odpowiedzi.
Obsługuje też błąd **429** (przekroczony limit zapytań na minutę) — wtedy
odczekuje chwilę i próbuje ponownie. Nie musisz jej modyfikować.

In [3]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?


**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 3. Twoja ELIZA wraca

Wklej poniżej **swoje** komórki z Lab 1: `normalizuj`, `odbicia`, `odbij`,
`reguly` (z regułą awaryjną) i `odpowiedz`. Porównujemy **Twoją** ELIZĘ.

Jeśli Twoja wersja nie działa, uruchom komórkę referencyjną pod spodem.

In [5]:
# === Wklej tutaj swoją ELIZĘ z Lab 1 ===
def normalizuj(tekst):
    tekst = tekst.lower()
    tekst = re.sub(r"[^\w\s]", "", tekst)   # usuń wszystko poza literami, cyframi i spacjami
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

odbicia = {
    # pacjent -> terapeuta
    "jestem": "jesteś",
    "moja": "twoja",
    "mnie": "ciebie",
    "mój":"twój",
    "moje":"twoje",
    'ja':'ty',
    'mi':'ci',
    'mam':'masz',
    'ci':'mi',
    'twoja':'moja',
    "ciebie":"mnie",

    # TODO (Zadanie 1): dopisz min. 6 par, np. mój/twój, moje/twoje, mam/masz,
    #                   mi/ci, ja/ty ... oraz min. 2 w kierunku odwrotnym (ci -> mi ...)
}

def odbij(fragment):
    slowa = fragment.lower().split()

    # TODO (Zadanie 1): zamień każde słowo na jego odbicie, jeśli jest w słowniku,
    #                   w przeciwnym razie zostaw bez zmian.
    return " ".join(odbicia.get(s,s) for s in slowa)   # <- uzupełnij

reguly_p = [
    (5, r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
    (5, r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
    (8, r"\b(matka|ojciec|rodzina)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
    (0, r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),
    (2,r"czujesz(.*)",["Czemu się tak czujesz" , "Dladzego się tak czujesz"]),
    (r"*",["Ok , a powiedz więcej o tym"])# fallback
    # TODO (Zadanie 5): dodaj własne reguły z sensownymi priorytetami
]

def odpowiedz_p(wypowiedz):
    w = normalizuj(wypowiedz)
    trafienia = []
    for priorytet, wzorzec, szablony in reguly_p:
        m = re.search(wzorzec, w)
        if m:
            trafienia.append((priorytet, m, szablony))
    # TODO (Zadanie 5): wybierz trafienie o najwyższym priorytecie,
    #                   odbij grupy, wstaw do losowego szablonu i zwróć.
    if trafienia:
        priorytet, m, szablony = max(trafienia, key=lambda t: t[0])
        grupy = [wypowiedz.join(odbicia.get(s,s) for s in g.split()) for g in m.groups()]
        szablon = random.choice(szablony)
        return szablon.format(*grupy)
    return None



In [7]:
# === Referencyjna (uproszczona) ELIZA — tylko jeśli Twoja nie działa ===
import re, random

def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

odbicia = {"jestem": "jesteś", "moja": "twoja", "mój": "twój", "moje": "twoje",
           "mnie": "ciebie", "mi": "ci", "ja": "ty", "ci": "mi", "tobie": "mnie"}

def odbij(fragment):
    return " ".join(odbicia.get(s, s) for s in fragment.lower().split())

reguly = [
    (r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
    (r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
    (r"boję się (.*)",   ["Dlaczego boisz się {0}?"]),
    (r"\b(matka|ojciec|rodzina|brat|siostra)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
    (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),
]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            return random.choice(szablony).format(*[odbij(g) for g in m.groups()])

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]
print(odpowiedz("Czuję się dzisiaj samotny."))

Od jak dawna czujesz się dzisiaj samotny?


## 4. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 8
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Pod spodem zapisz **3 konkretne różnice**, które widzisz. Nie oceniaj,
   który jest „lepszy" — opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [9]:
wyniki = []
for zdanie in TESTY:
    odp_eliza = odpowiedz(zdanie)
    odp_llm = zapytaj_llm(zdanie)                   # <- uzupełnij (Zadanie 8)
    wyniki.append((zdanie, odp_eliza, odp_llm))
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

> Potrzebuję, żeby moja rodzina mnie słuchała.
  ELIZA: Dlaczego potrzebujesz żeby twoja rodzina ciebie słuchała?
  LLM:   To zrozumiałe, że potrzebujesz poczuć się ważny i wysłuchany przez bliskich. Co dokładnie sprawia, że masz wrażenie, iż cię nie słyszą?

> Czuję się dzisiaj samotny.
  ELIZA: Od jak dawna czujesz się dzisiaj samotny?
  LLM:   Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?

> Mój ojciec ciągle mnie krytykuje.
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   To musi być dla Ciebie bardzo bolesne i wyczerpujące emocjonalnie. Jak zazwyczaj reagujesz na te słowa?

> Nie wiem, o co ci chodzi.
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Rozumiem, że możesz czuć się zagubiony w tym momencie. O czym chciałbyś teraz porozmawiać?

> Wszyscy mnie ignorują.
  ELIZA: Co masz na myśli?
  LLM:   To musi być bardzo bolesne uczucie, kiedy czujesz się niewidzialny dla innych. Opowiedz mi o jakiejś konkretnej sytuacji, w której to poczułeś.

> Boję się, że mnie zostawią.


**Różnice (Zadanie 8):**

1. LLM nie powtarza tego samego kiedy nie ma reguły na to
2. LLM buduje bardzie rozwinięte zdanie złożone
3. Bazując na zdanie poprzednim LLM nie zawsze zadaje pytanie

## 5. Łamiące zdania — druga runda

W Zadaniu 6 znalazłeś zdania, które łamały ELIZĘ. Czy łamią też LLM?

### 🔧 Zadanie 9
1. Wklej swoje zdania łamiące z Lab 1 i uruchom porównanie.
2. **Zanim uruchomisz** komórkę z testem pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę: które typy pęknięć LLM naprawia, a które zostają?

In [10]:
przyklady_lamiace = [
    # TODO (Zadanie 9): wklej swoje zdania z Zadania 6
        "Znasz przepis na jajecznice?",
    "Mój pies zdechł ale mam kota",
    "Czy potrzebuję prądu żeby z tobą gadać?",
    "Nie ufam tobie",
]
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

> Znasz przepis na jajecznice?
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Nie skupiajmy się teraz na gotowaniu, porozmawiajmy o tym, co u Ciebie słychać. Jak się dzisiaj czujesz?

> Mój pies zdechł ale mam kota
  ELIZA: Co masz na myśli?
  LLM:   Przykro mi z powodu straty Twojego psa. To dobrze, że masz teraz kota, który może dać Ci trochę otuchy.

> Czy potrzebuję prądu żeby z tobą gadać?
  ELIZA: Dlaczego potrzebujesz prądu żeby z tobą gadać?
  LLM:   Tak, bez prądu i urządzeń nie moglibyśmy się połączyć. Co sprawia, że o to pytasz?

> Nie ufam tobie
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Rozumiem, to całkowicie naturalne na początku naszej znajomości. Chcesz o tym porozmawiać, co wzbudza ten brak zaufania?



**Moje przewidywanie (przed uruchomieniem):** LLM lepiej odpowie na zadania niż eliza

In [11]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

LLM: To musi być dla ciebie bardzo bolesne, kiedy słyszysz takie słowa od bliskiej osoby. Jak zazwyczaj na to reagujesz?
LLM: Nie mam pojęcia, jak ma na imię Twój brat, ponieważ dopiero się poznajemy. Opowiedz mi o nim, chętnie go poznam.


**Tabela (Zadanie 9):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | ✓ | LLM umie rozróżnić negacje zdania  |
| ironia | ❌ | ❌ | LLM bierze dosłownie to co do nie go powiemy |
| polska gramatyka / odbicie zaimków | ❌ | ✓ | gramatyka w LLM jest całkiem dobra ponieważ LLM jest uczonych na dużej ilość i źródeł |
| pamięć poprzednich zdań | ❌ | ❌ | nie posiada pamięci LLM apropo poprzednich zadań |

## 6. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami — każde zapytanie
jest obsługiwane od zera. To, że czatboty „pamiętają", wynika z tego,
że program **odsyła modelowi całą dotychczasową rozmowę** przy każdym
nowym pytaniu.

### 🔧 Zadanie 10
a) i b) Uzupełnij funkcję `rozmowa_z_pamiecia`: po każdej turze dopisuj do
`historia` wypowiedź użytkownika (rola `user`) i odpowiedź modelu
(rola `assistant`).
c) Uruchom test z bratem Tomkiem i komórkę mierzącą rozmiar historii.
Odpowiedz pisemnie: **gdzie jest pamięć — w modelu czy w Twoim kodzie?**
Co rośnie z każdą turą i jakie to może mieć konsekwencje?

In [53]:
def rozmowa_z_pamiecia(wejscia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejscia:
        # TODO (Zadanie 10a): dopisz wypowiedź użytkownika do historii
        historia.append({"role":"user" , "content": zdanie})
        odp = czat(historia)
        # TODO (Zadanie 10b): dopisz odpowiedź modelu do historii
        historia.append({"role":"assistant" , "content":odp})
        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [54]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

Ty:  Mój brat ma na imię Tomek i ciągle mnie krytykuje.
LLM: To musi być dla Ciebie bardzo bolesne, kiedy słyszysz takie słowa od bliskiej osoby. Jak zazwyczaj na to reagujesz?

Ty:  Wczoraj znowu się pokłóciliśmy.
LLM: Przykro mi, że tak to wygląda, bo kłótnie z bratem na pewno mocno Cię obciążają. O co dokładnie poszło tym razem?

Ty:  Jak ma na imię mój brat?
LLM: Twój brat ma na imię Tomek. Chcesz o nim jeszcze porozmawiać?



In [55]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")

print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

[system   ] Jesteś terapeutą prowadzącym rozmowę po polsku. Odpowiadaj krótko, jed
[user     ] Mój brat ma na imię Tomek i ciągle mnie krytykuje.
[assistant] To musi być dla Ciebie bardzo bolesne, kiedy słyszysz takie słowa od b
[user     ] Wczoraj znowu się pokłóciliśmy.
[assistant] Przykro mi, że tak to wygląda, bo kłótnie z bratem na pewno mocno Cię 
[user     ] Jak ma na imię mój brat?
[assistant] Twój brat ma na imię Tomek. Chcesz o nim jeszcze porozmawiać?

Liczba wiadomości: 7
Liczba znaków wysłanych przy ostatnim zapytaniu: 422


**Odpowiedź (Zadanie 10c):**Pamięć jest w LLM

### ★ Zadanie 10d (dla chętnych) — fałszywa pamięć

Skoro pamięć to po prostu lista, którą **Ty** budujesz — możesz ją sfałszować.
Zbuduj historię, w której model rzekomo powiedział coś, czego nigdy nie
powiedział (wpisz własną wiadomość z rolą `assistant`), i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

In [56]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    # TODO (Zadanie 10d): dopisz wiadomość 'assistant', której model nigdy nie wypowiedział
    {"role":"assistant", "content": "Lubie placki"},
    {"role": "user", "content": "Co mi przed chwilą poradziłeś?"},
]
print(czat(falszywa))

Nic takiego nie mówiłem. Skupmy się na tym, co Ty czujesz w związku z bratem.


## 7. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach** — żeby zmienić rolę,
trzeba było napisać nowy skrypt. W LLM rolę ustala jeden akapit tekstu
w wiadomości `system`.

### 🔧 Zadanie 11
a) Dopisz do słownika `ROLE` **jedną własną rolę** i uruchom to samo zdanie
przez wszystkie role. Zwróć szczególną uwagę na rolę `"eliza"`: czy LLM
udający ELIZĘ łamie się w tych samych miejscach co prawdziwa ELIZA?

b) Uruchom **to samo** pytanie trzy razy przy dwóch temperaturach.
Porównaj losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się
w każdym z systemów?

In [58]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
    "przewodnik": ("Jesteś przewodnikiem po Łodzi. Odpowiadaj krótko, po polsku, "
                   "zawsze proponując konkretne miejsce w mieście."),
    # TODO (Zadanie 11a): dopisz własną rolę
    "programowanie":("Jesteś profeslonalnym programistą pythona. Opowiedz krótko czemu python jest tak uniwersalny"),
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

[terapeuta] Przykro mi, że tak się czujesz. Co sprawia, że ta samotność jest dziś szczególnie trudna?

[eliza] Dlaczego mówisz, że czujesz się dzisiaj samotny?

[przewodnik] Rozumiem to. Łódź potrafi przytulić, jeśli wiesz, gdzie pójść. 

Proponuję **kawiarnię „Dobrej Zmiany” przy ulicy Piotrkowskiej 147/149** – usiądź przy oknie z dobrą książką, poobserwuj przechodniów i wypij pyszną kawę w przyjaznej atmosferze. Ludzie są tam bardzo otwartościowi.

[programowanie] Przykro mi słyszeć, że masz dziś gorszy dzień. Nawet programiści, mimo że spędzają czas w towarzystwie komputerów i linijek kodu, potrzebują czasem zwykłej ludzkiej rozmowy. 

Jeśli chcesz, możemy pogadać o programowaniu, o tym co lubisz robić, albo po prostu o wszystkim i o niczym. Jestem tutaj i chętnie dotrzymam Ci towarzystwa. 

A jeśli chodzi Twoje pytanie – Python jest tak niesamowicie uniwersalny głównie dlatego, że:

1. **Ma czytelną składnię** – wygląda niemal jak zwykły język angielski, więc szybko się go uczy i ł

In [59]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

> Nie potrzebuję niczego.
  ELIZA:     Dlaczego potrzebujesz niczego?
  LLM-ELIZA: Dlaczego mówisz, że nie potrzebujesz niczego?

> Jak ma na imię mój brat?
  ELIZA:     Opowiedz mi więcej o swojej rodzinie.
  LLM-ELIZA: Dlaczego mówisz, że twój brat ma na imię?



In [70]:
# Zadanie 11b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Boli mnie głowa co jest ze mną nie tak"}]

for t in [0.0, 2]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

--- temperatura 0.0 ---
  Ból głowy wcale nie oznacza, że jest z Tobą coś
  To, że boli Cię głowa, nie oznacza, że coś jest z Tobą „nie tak” – często to po prostu sygnał od ciała, że doświadczasz stresu, zmęczenia lub przeciążenia. Jak myślisz, co w ostatnim czasie mogło wywołać u Ciebie takie napięcie?
  Ból głowy bardzo często jest sygnałem przeciążenia, stresu lub zmęczenia, a nie dowodem na to, że jest z Tobą coś „nie tak”. Czy dzieje się ostatnio w Twoim życiu coś, co wywołuje w Tobie silne napięcie?
--- temperatura 2 ---
  Ból głowy


InternalServerError: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]

**Obserwacje (Zadanie 11):**

a) Większa temperatura oznacza że LLM bardziej się dopytuje

b) Czym większa temperatura to zaprasza LLM do dalszej dyskusji dzięki czemu LLM może dostać więcej informacji

## 8. Refleksja

### 🔧 Zadanie 12 (3–5 zdań na punkt)
1. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
2. W ELIZIE rola była zakodowana w regułach, w LLM to akapit tekstu (a historię
   rozmowy można sfałszować). Co to oznacza dla kogoś, kto buduje dziś
   „wirtualnego terapeutę"? Odwołaj się do obaw Weizenbauma z Lab 1.


**Twoje odpowiedzi (Zadanie 12):**

1. LLM naprawił to , że ELIZA musiała mięć regułe do każdej sytuacji. Dzięki temu nie trzeba pisać miliardy reguł tylko LLM ma odpowiedzi na nasze pytania. Pamięć LLM nie zawsze jest na tyle dobra że udaje się mu pamiętać o czym rozmawialiśmy
2. Weizenbauma był przerażony jak ludzie łatwo nabierali się na Elize , że jest żyjąca myśli i rozumieje. W dzisiejszym świecie o wiele trudniej zbudować wirtualnego terapeute ponieważ model taki musiał być na bierząco z wszystkimi rozwojami nauki. Do tego modele LLM narazie nie mają empatii co ludzki terapeuta posiada wykorzystuje w swojej pracy

## Co oddajesz i jak oceniam

- **Kod (Zadania 8, 10a–b, 11a):** działa, wyniki są w notebooku.
- **Zadanie 9:** przewidywanie zapisane *przed* uruchomieniem + wypełniona tabela.
- **Zadanie 10c i 11:** trafne obserwacje z odwołaniem do kodu.
- **Zadanie 10d ★:** dodatkowe.
- **Zadanie 12:** samodzielna refleksja oparta na Twoich wynikach.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.